<a href="https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w5_reranking/llm_260408_cross_encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 260408 Reranking 심화: Score Filtering, Context Compression, MMR

**w5 day 2** - 리랭킹 파이프라인 완성

### 오늘 배우는 것
1. **Score Filtering** - 리랭킹 후 점수 기반으로 불필요한 문서 걸러내기 (fixed / dynamic / score gap / adaptive)
2. **Context Compression** - 문서를 압축해서 토큰 절약하기 (키워드 기반, LLM 기반, LangChain 내장)
3. **MMR (Maximum Marginal Relevance)** - 다양성을 고려한 압축
4. **평가 지표** - mAP, ILS로 리랭킹 효과 정량 측정

### 전체 흐름
```
검색(Retriever) -> 후보 문서 -> Re-rank -> Score Filter -> Compress -> Generator(LLM)
```

> **w5d1 복습**: Bi-encoder(쿼리/문서 각각 임베딩) vs Cross-encoder(쿼리+문서 함께 임베딩), 키워드/BM25/LLM 리랭킹, Pointwise vs Listwise

## 0. 환경 설정

In [1]:
!pip install -q faiss-cpu langchain langchain-community langchain-core langchain-openai matplotlib numpy openai pandas rank-bm25 scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from scipy.stats import norm

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [3]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

# load_dotenv()

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 데이터 준비 + 벡터 검색 (w5d1 복습)

w5d1에서 만든 문서 8개와 벡터스토어를 그대로 사용합니다.

In [4]:
# 샘플 문서 8개 - AI/NLP 관련 지식
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]

In [5]:
# FAISS 벡터스토어 + BM25 리트리버 생성
vectorstore = FAISS.from_documents(documents, embeddings_model)
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [6]:
# 임베딩 캐시 + 헬퍼 함수
doc_embeddings = {}

def get_embedding(text):
    """텍스트를 임베딩 벡터로 변환"""
    return np.array(embeddings_model.embed_query(text))

for doc in documents:  # 각 문서의 임베딩을 미리 캐싱
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

In [7]:
# 코사인 유사도 헬퍼 함수
def cosine_sim(emb_a, emb_b):
    """두 임베딩 벡터 간 코사인 유사도 계산"""
    return np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))

In [8]:
# 벡터 검색 함수 (w5d1에서 만든 것)
# FAISS는 L2 거리를 반환하므로, 1/(1+distance)로 유사도 점수로 변환
def vector_search(query, vectorstore, top_k=5):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc, 1.0 / (1.0 + score)) for doc, score in results]

query = "트랜스포머와 BERT의 관계"
results = vector_search(query, vectorstore)
results

[(Document(id='55462d4d-b767-4f92-91fe-a95f17211799', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.52375495)),
 (Document(id='3a72a5a8-4d75-4b19-ad64-ee7e4ff28d18', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.46395132)),
 (Document(id='99d0e373-9b18-4305-a677-0da697d88f46', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41890207)),
 (Document(id='2bd456d6-918f-4a1b-8fae-79e7dc35bba1', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.41281974)),
 (Document(id='6616eece-4ab2-447a-bf59-b71ced7c0117', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39723063))]

## 2. Score Filtering - 점수 기반 문서 필터링

리랭킹을 했으면 이제 **점수가 낮은 문서는 걸러내야** 합니다.  
Generator(LLM)에 넘기기 전에 불필요한 문서를 제거하는 단계입니다.

> **비유**: 면접관이 서류 심사(1차 검색) 후 면접(리랭킹)을 봤는데,  
> 면접 점수가 너무 낮은 사람까지 최종 합격시키면 안 되겠죠?  
> Score Filter는 '커트라인'을 정하는 것입니다.

### 4가지 필터링 전략
| 방법 | 설명 | 장점 |
|------|------|------|
| **Top-K** | 상위 K개만 선택 | 가장 단순 |
| **Fixed Threshold** | 고정 점수 이상만 (예: 0.5 이상) | 직관적 |
| **Dynamic Threshold** | 평균 - n*표준편차 기준 | 분포에 따라 자동 조절 |
| **Score Gap** | 점수 급락 지점에서 자르기 | 자연스러운 경계 |

> **@staticmethod 참고**: 아래 클래스에서 `self` 없이 함수를 정의합니다.  
> 객체를 만들지 않고 `ScoreFilter.fixed_threshold(...)` 처럼 바로 호출할 수 있어요.  
> 관련 함수들을 하나의 클래스 안에 묶어서 코드 가독성을 높이는 패턴입니다.

In [9]:
class ScoreFilter:
    """점수 기반 문서 필터링 전략 모음"""

    @staticmethod
    def fixed_threshold(scored_docs, threshold=0.5):
        """고정 커트라인: threshold 이상인 문서만 통과"""
        return [(doc, s) for doc, s in scored_docs if s >= threshold]

    @staticmethod
    def dynamic_threshold(scored_docs, std_factor):
        """동적 커트라인: 평균 - (std_factor * 표준편차) 이상만 통과
        점수 분포에 따라 자동으로 커트라인이 조절됨"""
        scores = [s for _, s in scored_docs]
        threshold = np.mean(scores) - std_factor * np.std(scores)
        return [(doc, s) for doc, s in scored_docs if s >= threshold]

    @staticmethod
    def score_gap(scored_docs, min_docs=2):
        """점수 급락 지점에서 자르기
        비유: 0.9, 0.85, 0.7 ... 0.2 일 때 0.7->0.2 구간이 가장 크게 떨어지므로 거기서 자름"""
        if len(scored_docs) <= min_docs:
            return scored_docs

        scores = [s for _, s in scored_docs]
        # 인접한 점수들의 차이(gap)를 계산
        gaps = [(scores[i] - scores[i+1], i+1) for i in range(len(scores)-1)]
        # 가장 큰 gap이 있는 위치를 찾음
        max_gap_idx = max(gaps, key=lambda x: x[0])[1]
        cut = max(min_docs, max_gap_idx)  # 최소 min_docs개는 보장
        return scored_docs[:cut]

In [10]:
# 테스트용 더미 데이터 (점수가 뚜렷하게 구분되는 예시)
test_scores = [
    (Document(page_content="", metadata={'id': f'd{i}'}), s)
    for i, s in enumerate([0.9, 0.85, 0.7, 0.5, 0.2])
]
test_scores

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5),
 (Document(metadata={'id': 'd4'}, page_content=''), 0.2)]

In [11]:
# Fixed threshold: 0.5 이상만 통과 -> d4(0.2)만 탈락
filtered = ScoreFilter.fixed_threshold(test_scores, threshold=0.5)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

In [12]:
# Dynamic threshold: 평균 - 1*표준편차 이상만 통과
# 점수 분포가 넓으면 커트라인이 내려가고, 좁으면 올라감
filtered = ScoreFilter.dynamic_threshold(test_scores, std_factor=1)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

In [13]:
# Score gap: 점수가 갑자기 떨어지는 곳에서 자름
# 0.5 -> 0.2 구간이 gap=0.3으로 가장 크므로 거기서 자름
filtered = ScoreFilter.score_gap(test_scores, min_docs=2)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

### AdaptiveFilter - 자동 전략 선택

점수 분포를 보고 어떤 필터링 전략을 쓸지 **자동으로** 결정합니다.

| 조건 | 선택되는 전략 | 이유 |
|------|-------------|------|
| 표준편차 > 0.2 | Score Gap | 점수 편차가 크면 자연스러운 끊김이 있을 가능성 높음 |
| 점수 범위 < 0.1 | Top 50% | 점수가 다 비슷하면 상위 절반만 |
| 그 외 | Dynamic Threshold | 일반적인 경우 |

In [14]:
class AdaptiveFilter:
    """점수 분포에 따라 필터링 전략을 자동으로 선택"""

    @staticmethod
    def adapt_filter(scored_docs):
        scores = [s for _, s in scored_docs]
        std = np.std(scores)                        # 표준편차
        score_range = max(scores) - min(scores)     # 최대-최소 범위

        if std > 0.2:
            # 점수 편차가 크면 -> score gap으로 자연스러운 경계 찾기
            result = ScoreFilter.score_gap(scored_docs, min_docs=2)
        elif score_range < 0.1:
            # 점수가 다 비슷하면 -> 상위 50%만 (top-k와 같은 효과)
            result = scored_docs[:len(scored_docs)//2]
        else:
            # 일반적인 경우 -> dynamic threshold
            result = ScoreFilter.dynamic_threshold(scored_docs, std_factor=1)

        return result

In [15]:
# 테스트: std=0.26 > 0.2 이므로 score_gap 전략이 선택됨
filtered = AdaptiveFilter.adapt_filter(test_scores)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

## 3. Context Compression - 문서 압축

리랭킹 + 필터링으로 좋은 문서를 골랐어도, 그 문서들이 **너무 길면** 토큰을 낭비합니다.  
GPT-4o-mini의 context window가 128K 토큰이지만, 여기에는 시스템 프롬프트 + 대화 맥락 + 유저 질문 + 답변 공간이 모두 들어가야 합니다.

> **비유**: 시험 공부할 때 교과서 전체를 읽는 것보다,  
> 핵심만 정리한 요약 노트로 공부하는 게 효율적인 것과 같습니다.  
> 다만 요약할 때 중요한 키워드가 빠지면 안 되겠죠?

### 압축 방법 3가지
1. **키워드 기반 추출** - 쿼리 키워드가 많이 포함된 문장만 추출 (빠르지만 단순)
2. **LLM 기반 요약** - LLM에게 요약 요청 (정확하지만 비용/시간)
3. **MMR 기반 압축** - 관련성 + 다양성을 모두 고려하여 문장 선택

### 3-1. 키워드 기반 추출 압축

쿼리의 키워드와 겹치는 단어가 많은 문장을 우선 선택합니다.  
`re.split(r'[.!?]\s*', document)` - 정규표현식으로 마침표/느낌표/물음표 기준 문장 분리

In [16]:
def extractive_compress(query, document, max_sentences=3):
    """키워드 기반 추출 압축: 쿼리와 키워드가 많이 겹치는 문장을 선택"""
    # 정규표현식으로 문장 분리 (.!? 뒤의 공백 포함)
    # \s* = 0개 이상의 공백
    sentences = re.split(r'[.!?]\s*', document)

    if not sentences:
        return document

    # 쿼리 키워드 집합 만들기
    query_terms = set(query.lower().split())

    # 각 문장별로 쿼리 키워드와의 겹침 수 계산
    scored = []
    for sent in sentences:
        sent_terms = set(sent.lower().split())
        overlap = len(query_terms & sent_terms)  # 교집합 크기
        scored.append((sent, overlap))

    # 겹침이 많은 순으로 정렬, 상위 max_sentences개 선택
    scored.sort(key=lambda x: x[1], reverse=True)
    selected = [s for s, _ in scored[:max_sentences]]

    return ". ".join(selected) + "."

In [17]:
# 테스트: 긴 문서를 3문장으로 압축
long_doc = "트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다. 자연어 처리뿐 아니라 컴퓨터 비전에서도 활용됩니다"
query = "트랜스포머와 BERT의 관계"

compressed = extractive_compress(query, long_doc, max_sentences=3)
print(compressed)
# 참고: 한국어는 스페이스 기준 split이 형태소 분석보다 부정확해서 결과가 완벽하지 않을 수 있음

트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.


### 3-2. LLM 기반 요약 압축

LLM에게 "쿼리에 답하는 데 필요한 핵심만 추출해줘"라고 요청합니다.  
키워드 기반보다 훨씬 정확하지만, API 호출 비용과 시간이 듭니다.

In [18]:
# LLM 압축 체인: LCEL 패턴 (prompt | llm | parser)
compress_chain = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서 요약 전문가입니다"),
    ("human", """다음 문서에서 쿼리에 답하는데 필요한 핵심 정보만 추출하세요.
    불필요한 내용은 제거하고, {max_tokens}자 이내로 압축하세요.

    쿼리 : {query}
    문서 : {document}

    압축결과:""")
]) | llm | StrOutputParser()

In [19]:
def llm_compress(query, document, max_tokens=100):
    """LLM을 사용해서 문서를 쿼리 중심으로 압축"""
    return compress_chain.invoke({
        'query': query,
        'document': document,
        'max_tokens': max_tokens
    })

In [20]:
compressed_llm = llm_compress(query, long_doc)
print(compressed_llm)
# 예시 결과: "트랜스포머는 2017년 발표된 아키텍처로, BERT와 GPT의 기초가 됩니다."

트랜스포머는 2017년 구글이 발표한 아키텍처로, BERT와 GPT의 기반이 됩니다.


### 3-3. LangChain 내장 ContextualCompressionRetriever

LangChain에서는 리트리버 + 압축을 한 번에 해주는 `ContextualCompressionRetriever`를 제공합니다.  
내부적으로: base_retriever로 문서를 가져온 뒤 -> base_compressor로 압축

In [21]:
# LLMChainExtractor: LLM을 사용해서 관련 부분만 추출하는 압축기
compressor = LLMChainExtractor.from_llm(llm)

# 압축 리트리버 = 베이스 리트리버(FAISS) + 압축기(LLM)
compressor_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={'k': 3})
)

In [22]:
# 쿼리를 넣으면 리트리버 + 압축이 한 번에 수행됨
# (일반 리트리버보다 시간이 더 걸림 - 압축 단계가 추가되므로)
compressed_docs = compressor_retriever.invoke(query)
compressed_docs
# 3개 검색 -> 쿼리와 관련 없는 내용은 제거되어 2개만 남을 수 있음

[Document(metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
 Document(metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.')]

### 3-4. 압축 품질 평가

압축이 잘 됐는지 3가지 기준으로 평가합니다:
1. **압축률** - 원본 대비 얼마나 줄었는지 (낮을수록 많이 압축)
2. **키워드 보존율** - 쿼리 관련 키워드가 살아있는지 (높을수록 좋음)
3. **의미 보존도** - 원본과 압축본의 임베딩 유사도 (높을수록 의미 잘 보존)

In [23]:
def evaluate_compression(original, compressed, query):
    """압축 품질을 3가지 지표로 평가"""
    # 1) 압축률: 압축본 길이 / 원본 길이
    ratio = len(compressed) / len(original)

    # 2) 키워드 보존율: 쿼리 키워드가 압축 후에도 남아있는 비율
    query_terms = set(query.lower().split())
    orig_terms = set(original.lower().split())
    comp_terms = set(compressed.lower().split())

    orig_query_terms = query_terms & orig_terms    # 원본에 있는 쿼리 키워드
    comp_query_terms = query_terms & comp_terms    # 압축본에 있는 쿼리 키워드
    keyword_coverage = len(comp_query_terms) / len(orig_query_terms) if orig_query_terms else 1.0

    # 3) 의미 보존도: 원본과 압축본의 임베딩 코사인 유사도
    original_emb = np.array(embeddings_model.embed_query(original))
    compressed_emb = get_embedding(compressed)
    semantic_similarity = cosine_sim(original_emb, compressed_emb)

    return {
        'compression_ratio': ratio,           # 낮을수록 많이 압축
        'keyword_coverage': keyword_coverage,  # 높을수록 좋음
        'semantic_similarity': semantic_similarity  # 높을수록 좋음
    }

In [24]:
# LLM 압축 결과 평가
evaluate_compression(long_doc, compressed_llm, query)
# 예시: compression_ratio=0.29 (71% 압축), keyword_coverage=1.0, semantic_similarity=0.74

{'compression_ratio': 0.3197278911564626,
 'keyword_coverage': 1.0,
 'semantic_similarity': np.float64(0.779209799657259)}

## 4. MMR 기반 압축 - 다양성을 고려한 문장 선택

**MMR (Maximum Marginal Relevance)** 은 관련성(Relevance)과 다양성(Diversity)의 균형을 잡는 알고리즘입니다.

> **비유**: 뷔페에서 음식을 고를 때,  
> 내가 좋아하는 것(관련성)만 고르면 비슷한 것만 잔뜩 담게 됩니다.  
> MMR은 "좋아하는 것 + 아직 안 담은 종류"를 균형있게 고르는 전략입니다.

### MMR 수식
```
MMR = p * Relevance(s, query) - (1-p) * max(Similarity(s, selected))
```
- `p` (lambda): 관련성 vs 다양성 비율 파라미터
  - p=0.7 -> 관련성 70%, 다양성 30% (관련된 것 위주)
  - p=0.3 -> 관련성 30%, 다양성 70% (다양한 것 위주)
- `Relevance(s, query)`: 후보 문장 s와 쿼리의 유사도
- `max(Similarity(s, selected))`: 후보 문장 s와 이미 선택된 문장들 중 가장 유사한 것과의 유사도 (높으면 = 이미 비슷한 게 있다 = 다양성이 낮다)

In [25]:
def mmr_compress(query, document, max_sentences=3, param=0.5):
    """MMR 기반 압축: 관련성 + 다양성을 모두 고려하여 문장 선택
    param: 관련성 비율 (1에 가까울수록 관련성 중시, 0에 가까울수록 다양성 중시)
    """
    # 문장 분리 + 너무 짧은 문장 제거
    sentences = re.split(r'[.!?]\s*', document)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    if len(sentences) <= max_sentences:
        return ". ".join(sentences) + "."

    # 쿼리와 각 문장의 임베딩 준비
    query_emb = get_embedding(query)
    sent_embs = [get_embedding(s) for s in sentences]

    selected = []       # 선택된 문장 인덱스
    remaining = list(range(len(sentences)))  # 아직 선택 안 된 문장 인덱스

    for _ in range(max_sentences):
        best_score = -float('inf')  # 가능한 가장 작은 값으로 초기화
        best_idx = -1

        for idx in remaining:
            # (1) 쿼리와의 관련성
            relevance = cosine_sim(query_emb, sent_embs[idx])

            # (2) 이미 선택된 문장들과의 최대 유사도 (높으면 = 중복)
            if selected:
                max_sim = max(cosine_sim(sent_embs[idx], sent_embs[s]) for s in selected)
            else:
                max_sim = 0.0  # 첫 번째 선택시에는 다양성 패널티 없음

            # MMR 점수 = 관련성 - 중복 패널티
            mmr = param * relevance - (1 - param) * max_sim

            if mmr > best_score:
                best_score = mmr
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    # 원래 순서대로 정렬해서 반환 (문맥 유지)
    return ". ".join(sentences[i] for i in sorted(selected)) + "."

In [26]:
# param=0.7: 관련성 70% -> 쿼리와 관련된 문장 위주로 선택
print("param=0.7 (관련성 중시):")
print(mmr_compress(query, long_doc, max_sentences=2, param=0.7))
print()

# param=0.3: 다양성 70% -> 서로 다른 내용의 문장을 골고루 선택
print("param=0.3 (다양성 중시):")
print(mmr_compress(query, long_doc, max_sentences=2, param=0.3))

param=0.7 (관련성 중시):
트랜스포머는 2017년 구글이 발표한 아키텍처입니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다.

param=0.3 (다양성 중시):
트랜스포머는 2017년 구글이 발표한 아키텍처입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.


## 5. 리랭킹 효과 평가 - mAP, ILS

리랭킹이 실제로 효과가 있었는지 정량적으로 측정합니다.

### mAP (mean Average Precision)
- **Precision@k**: 상위 k개 중 관련 문서 비율
- **AP (Average Precision)**: 관련 문서가 나올 때마다의 Precision을 평균
- **mAP**: 여러 쿼리의 AP를 평균

> **비유**: 검색 결과에서 관련 문서가 **위쪽에 몰려있을수록** AP가 높아집니다.  
> [관련O, 관련O, 관련X, 관련X] 가 [관련X, 관련X, 관련O, 관련O] 보다 AP가 높습니다.

**예시 계산:**
```
검색 결과: [관련O, 관련X, 관련O, 관련X, 관련O]
Precision:  1/1       -     2/3      -     3/5
AP = (1/1 + 2/3 + 3/5) / 3 = 0.756
```

### ILS (Intra-List Similarity)
- 상위 문서들 **간의** 유사도 평균
- 높으면 = 비슷한 문서만 모임 = 다양성이 낮음
- 낮으면 = 다양한 문서가 섞임 = 다양성이 높음

In [27]:
def average_precision(retrieved_ids, relevant_ids):
    """단일 쿼리에 대한 Average Precision 계산
    retrieved_ids: 검색된 문서 ID 리스트 (순서 중요!)
    relevant_ids: 정답(관련) 문서 ID 집합
    """
    relevant_set = set(relevant_ids)
    hits = 0           # 지금까지 나온 관련 문서 수
    sum_precision = 0.0

    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_set:
            hits += 1
            precision_at_i = hits / (i + 1)  # 현재 위치에서의 precision
            sum_precision += precision_at_i

    # 관련 문서 수로 나누어 평균
    return sum_precision / len(relevant_set) if relevant_set else 0.0

In [28]:
def mean_average_precision(query_results):
    """여러 쿼리의 AP를 평균 -> mAP
    query_results: {query: (retrieved_ids, relevant_ids), ...}
    """
    aps = []
    for query, (retrieved, relevant) in query_results.items():
        ap = average_precision(retrieved, relevant)
        aps.append(ap)

    map_score = np.mean(aps)
    return map_score

In [29]:
# 벡터 검색 결과의 AP 테스트
relevant = {'d1', 'd2', 'd3'}  # 정답: 트랜스포머, BERT, GPT 관련 문서
orig_order = [doc.metadata['id'] for doc, _ in results]
print("검색 순서:", orig_order)

ap = average_precision(orig_order, relevant)
print(f"AP = {ap}")
# d1, d2, d3가 1,2,3위에 있으므로 AP = 1.0 (완벽한 순위)

검색 순서: ['d1', 'd2', 'd3', 'd6', 'd7']
AP = 1.0


In [30]:
def intra_list_similarity(doc_ids, doc_embeddings, top_k=5):
    """상위 문서들 간의 평균 유사도 (다양성 지표)
    높을수록 다양성이 낮음 (비슷한 문서끼리 모여있음)
    """
    ids = doc_ids[:top_k]
    embs = [doc_embeddings[did] for did in ids if did in doc_embeddings]

    if len(embs) < 2:
        return 0.0

    # 모든 쌍의 코사인 유사도를 계산
    # (a,b), (a,c), (a,d), (b,c), (b,d), (c,d) ...
    sims = []
    for i in range(len(embs)):
        for j in range(i + 1, len(embs)):
            sims.append(cosine_sim(embs[i], embs[j]))

    return np.mean(sims)

In [31]:
# ILS 테스트
ils_score = intra_list_similarity(orig_order, doc_embeddings)
print(f"ILS = {ils_score}")
# 상위 문서들이 서로 얼마나 비슷한지 확인

ILS = 0.3413383419868966


## 6. (w5d1 복습) 리랭킹 함수들

w5d1에서 만든 리랭킹 함수들을 여기서도 사용합니다.  
위의 Score Filter, Compression, 평가 지표와 조합해서 전체 파이프라인을 구성할 수 있습니다.

In [32]:
# --- w5d1 리랭킹 함수 (참고용) ---

# 키워드 리랭킹: 쿼리 키워드가 문서에 몇 개 있는지로 점수 보정
def keyword_rerank(query, search_results):
    query_terms = set(query.lower().split())
    reranked = []
    for doc, orig_score in search_results:
        doc_terms = doc.page_content.lower().split()
        keyword_hits = sum(1 for t in doc_terms if t in query_terms)
        new_score = orig_score + 0.1 * keyword_hits
        reranked.append((doc, new_score, orig_score))
    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked

# 순위 변동 확인: 리랭킹 전후 순위 비교표
def rank_change(original_results, reranked_results):
    orig_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _) in enumerate(original_results)}
    new_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _, _) in enumerate(reranked_results)}
    changes = []
    for doc_id in orig_ranks:
        old_r = orig_ranks[doc_id]
        new_r = new_ranks.get(doc_id, -1)
        changes.append({'doc_id': doc_id, 'before': old_r, 'after': new_r, 'change': old_r - new_r})
    return pd.DataFrame(changes).sort_values('after')

In [33]:
# BM25 리랭커: TF-IDF 기반 점수로 리랭킹 (w5d1)
class BM25Reranker:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1  # term frequency 가중치
        self.b = b     # 문서 길이 보정 파라미터

    def rerank(self, query, search_results):
        docs = [doc for doc, _ in search_results]
        tokenized = [doc.page_content.lower().split() for doc in docs]
        avg_dl = np.mean([len(t) for t in tokenized])  # 평균 문서 길이

        # Document Frequency 계산
        df_count = Counter()
        for tokens in tokenized:
            for t in set(tokens):
                df_count[t] += 1
        N = len(docs)

        query_tokens = query.lower().split()
        scored = []

        for i, doc in enumerate(docs):
            score = 0.0
            doc_len = len(tokenized[i])
            tf_count = Counter(tokenized[i])

            for qt in query_tokens:
                tf = tf_count.get(qt, 0)
                if tf == 0:
                    continue
                df = df_count.get(qt, 0)
                # IDF: 희귀한 단어일수록 가중치 높음
                idf = math.log((N - df + 0.5) / (df + 0.5) + 1)
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / avg_dl)
                score += idf * numerator / denominator

            scored.append((doc, score))

        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

In [34]:
# LLM Pointwise 리랭킹: LLM이 각 문서에 0.0~1.0 점수 부여 (w5d1)
def llm_rerank(query, search_results, top_k=3):
    docs_text = '\n'.join(
        f"[{doc.metadata['id']}] {doc.page_content}" for doc, _ in search_results
    )
    score_template = ", ".join(f'"{doc.metadata["id"]}": 0.0-1.0' for doc, _ in search_results)

    scoring_chain = ChatPromptTemplate.from_messages([
        ('system', '당신은 scoring 시스템 입니다. 항상 json 형태로 출력하세요'),
        ('human', """다음 쿼리에 대해 각 문서의 관련성을 0.0~1.0으로 평가하세요.

        쿼리 : {query}

        문서들 :
        {docs_text}

        "스트링으로 묶지말고" JSON으로 답하세요:
        {{"scores" : {{{score_template}}}}}""")
    ]) | llm | StrOutputParser()

    result = scoring_chain.invoke({
        'query': query, 'docs_text': docs_text, 'score_template': score_template
    })

    cleaned = result.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned.replace('```json', '').replace('```', '')
    parsed = json.loads(cleaned)
    scores = parsed.get('scores', {})

    scored = []
    for doc, orig in search_results:
        rerank_score = scores.get(doc.metadata['id'], 0.0)
        scored.append((doc, float(rerank_score), orig))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

In [35]:
# LLM Listwise 리랭킹: LLM이 문서 전체를 보고 순서를 정함 (w5d1)
def llm_listwise_rerank(query, search_results):
    docs_text = '\n'.join(
        f"{i+1}. [{doc.metadata['id']}] {doc.page_content[:80]}"
        for i, (doc, _) in enumerate(search_results)
    )

    ranking_chain = ChatPromptTemplate.from_messages([
        ('system', "당신은 문서 랭킹 시스템입니다"),
        ('human', """다음 문서들을 쿼리와의 관련성 순서로 정렬하세요.

        쿼리 : {query}

        문서들 : {docs_text}

        가장 관련성 높은 순서대로 문서 번호를 쉼표로 나열하세요 (예: 3,1,5,2,4):""")
    ]) | llm | StrOutputParser()

    result = ranking_chain.invoke({'query': query, 'docs_text': docs_text})
    order = [int(x.strip()) for x in result.strip().split(',')]

    docs_list = [doc for doc, _ in search_results]
    reranked = []
    for rank, idx in enumerate(order):
        if 1 <= idx <= len(docs_list):
            reranked.append((docs_list[idx-1], 1.0 - rank * 0.1))
    return reranked

In [36]:
# Hybrid 리랭킹: BM25 + LLM 점수를 가중 합산 (w5d1)
def hybrid_rerank(query, search_results, bm25_weight=0.3):
    bm25 = BM25Reranker()
    bm25_scored = bm25.rerank(query, search_results)
    bm25_map = {doc.metadata['id']: score for doc, score in bm25_scored}

    llm_scored = llm_rerank(query, search_results, top_k=len(search_results))
    llm_map = {doc.metadata['id']: score for doc, score, _ in llm_scored}

    def normalize(scores_dict):
        vals = list(scores_dict.values())
        min_, max_ = min(vals), max(vals)
        range_ = max_ - min_ if max_ > min_ else 1e-8
        return {k: (v - min_) / range_ for k, v in scores_dict.items()}

    bm25_norm = normalize(bm25_map)
    llm_norm = normalize(llm_map)

    combined = []
    for doc, _ in search_results:
        did = doc.metadata['id']
        score = bm25_weight * bm25_norm.get(did, 0) + (1 - bm25_weight) * llm_norm.get(did, 0)
        combined.append((doc, score))
    combined.sort(key=lambda x: x[1], reverse=True)
    return combined

## 7. 오늘 핵심 정리

### 전체 RAG 파이프라인에서 오늘 배운 부분
```
Query -> [1차 Retriever] -> 후보 문서들
                              |
                         [Re-ranking]     <- w5d1 (키워드/BM25/LLM/Hybrid)
                              |
                        [Score Filter]    <- 오늘! (fixed/dynamic/gap/adaptive)
                              |
                      [Compression]       <- 오늘! (키워드/LLM/MMR)
                              |
                        [Generator]  ->  최종 답변
```

### Pointwise vs Listwise 리랭킹 (w5d1 복습)
| | Pointwise | Listwise |
|---|---|---|
| 방식 | 문서 1개씩 점수 매김 | 문서 N개를 한번에 정렬 |
| 장점 | 정확, 병렬 처리 가능 | 문서 간 비교 가능 |
| 단점 | 문서 간 비교 불가 | 할루시네이션, 컨텍스트 제한 |
| 적합 | 빠른 응답 필요 시 | 소수 문서 정밀 정렬 시 |

### 평가 지표
- **mAP**: 관련 문서가 상위에 있을수록 높음 (리랭킹 품질)
- **ILS**: 상위 문서들이 비슷할수록 높음 (다양성의 역수)
- **압축 평가**: 압축률 + 키워드 보존율 + 의미 보존도

### 하네스 엔지니어링 (수업 중 언급)
에이전트(Claude Code 등)에게 빡빡한 규칙을 줘서, 편리하면서도 안전하게 작업시키는 기법.  
Context compression 규칙도 하네스에 포함시키는 것이 좋음 (예: 토큰 80% 차면 압축).